# 05 – Fetal-head segmentation and HC measurement (Phase 2, M5)
1. Zero-shot MobileSAM baseline (GT-box and detector-box prompts), compared against the sonographer masks.
2. U-Net (MobileNetV2 encoder, 512×352 input) trained with Dice+BCE.
3. TEST evaluation: Dice/IoU/mask mAP/HD95, HC MAE with Bland–Altman, full detector→U-Net chain, failure gallery.
Acceptance: Dice ≥ 0.95 and HC MAE ≤ 3.0 mm on TEST. Requires notebooks 03 and 04 (detector weights + operating point).

## 0. Setup (Colab)
Clone the repo, install the pinned requirements, point `FETALVISION_DATA` at the data folder (Google Drive or a local copy) and log in to W&B (optional – offline mode is used when no key is present).

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone https://github.com/lamooooo1418/FetalVision.git  # TODO: replace with the team repository URL
    %cd FetalVision
    !pip install -q -r requirements.txt && pip install -q -e .
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["FETALVISION_DATA"] = "/content/drive/MyDrive/FetalVision_data"   # must contain hc18/raw/training_set.zip + CSV
else:
    os.environ.setdefault("FETALVISION_DATA", os.path.abspath("../data"))
    %cd ..  # notebook lives in notebooks/, CLIs run from the repo root
print("FETALVISION_DATA =", os.environ["FETALVISION_DATA"])

In [ ]:
# W&B login (optional). Without a key the CLIs log offline; sync later with `wandb sync wandb/offline-run-*`.
import os
if os.environ.get("WANDB_API_KEY"):
    import wandb; wandb.login()
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_MODE=offline")

In [ ]:
!python -m fetalvision.segmentation.sam_baseline --config configs/phase2_segmentation.yaml

In [ ]:
import json; m = json.load(open("results/phase2/segmentation_sam/metrics.json"))
print(json.dumps({s: {p: {k: m["metrics"][s][p][k] for k in ["dice_mean", "dice_std", "share_dice_lt_0.9", "hc_mae_mm"]} for p in ["gt_box", "det_box"]} for s in ["val", "test"]}, indent=1))
print(m["sam_vs_sonographer_agreement"]["statement"])

In [ ]:
from IPython.display import Image, display, JSON
import glob, json
for f in sorted(sum([glob.glob(g) for g in ['results/phase2/segmentation_sam/figures/sam_examples_test.png', 'results/phase2/segmentation_sam/figures/sam_worst3_test.png']], [])):
    print(f); display(Image(filename=f, width=900))

In [ ]:
!python -m fetalvision.segmentation.train_unet --config configs/phase2_segmentation.yaml

In [ ]:
from IPython.display import Image, display, JSON
import glob, json
for f in sorted(sum([glob.glob(g) for g in ['results/phase2/segmentation/figures/learning_curves.png']], [])):
    print(f); display(Image(filename=f, width=900))

In [ ]:
!python -m fetalvision.segmentation.evaluate --config configs/phase2_segmentation.yaml

In [ ]:
import json; m = json.load(open("results/phase2/segmentation/metrics.json"))["metrics"]
for k in ["unet_full_image", "unet_detector_crop_chain"]:
    print(k, "Dice", m[k]["dice"], "IoU mean", round(m[k]["iou"]["mean"], 4), "mask mAP", m[k]["mask_mAP"], "HC", m[k]["hc"])
print("SAM:", json.dumps(m["sam_baseline_test"], indent=1)); print("acceptance:", m["acceptance"])
print(json.dumps(json.load(open("results/phase2/segmentation/figures/captions.json"))["failure_cases_unet_full"], indent=1))

In [ ]:
from IPython.display import Image, display, JSON
import glob, json
for f in sorted(sum([glob.glob(g) for g in ['results/phase2/segmentation/figures/hc_bland_altman_unet_full.png', 'results/phase2/segmentation/figures/failure_gallery_unet_full.png', 'results/phase2/segmentation/figures/dice_histogram.png']], [])):
    print(f); display(Image(filename=f, width=900))